# 2. EDA và trực quan hóa – BRFSS2015 (diabetes.csv, nhãn nhị phân)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, os
os.makedirs("../figures", exist_ok=True)
sns.set_theme(style="whitegrid")
df = pd.read_csv("../data/raw/diabetes.csv")
T = "Diabetes_binary"

## 2.1 Kích thước, kiểu dữ liệu, giá trị thiếu, trùng lặp

In [ ]:
print("Số dòng, cột:", df.shape, "(Proposal: ~253.680 mẫu, 21 thuộc tính + 1 nhãn)")
print("Kiểu dữ liệu:", df.dtypes.value_counts().to_dict())
print("Giá trị thiếu:", int(df.isna().sum().sum()))
dup = df.duplicated().sum()
print(f"Dòng trùng lặp: {dup} ({dup/len(df):.2%})")

**Nhận xét:** 253.680 dòng, 22 cột (21 thuộc tính + nhãn) khớp Proposal; toàn bộ là int64, không thiếu giá trị. Có 24.206 dòng trùng (9,54%) nên phải xử lý trước khi chia train/test (xem bước 3).

## 2.2 Phân bố biến mục tiêu

In [ ]:
cnt = df[T].value_counts().sort_index()
pct = cnt / len(df) * 100
ax = sns.barplot(x=["0: Không","1: Tiền ĐTĐ/ĐTĐ"], y=cnt.values, color="#4C78A8")
for i,(c,p) in enumerate(zip(cnt.values,pct.values)): ax.text(i, c, f"{c:,}\n({p:.1f}%)", ha="center", va="bottom")
ax.set_ylabel("Số mẫu"); ax.set_title("Phân bố biến mục tiêu"); ax.set_ylim(0, cnt.max()*1.15)
plt.tight_layout(); plt.savefig("../figures/01_target_distribution.png", dpi=150); plt.show()

**Nhận xét:** Dữ liệu mất cân bằng: 86,1% không bệnh và 13,9% có bệnh (218.334 so với 35.346 mẫu). Con số 84% / 2% / 14% trong Proposal là của bản 3 lớp; ở bản nhị phân lớp 1 chính là lớp ĐTĐ 13,9%, còn tiền ĐTĐ (1,8%) nằm lẫn trong lớp 0. Cần class_weight/resampling và đánh giá bằng recall, F1, PR-AUC thay vì accuracy.

## 2.3 Histogram và boxplot: BMI, MentHlth, PhysHlth

In [ ]:
cols = ["BMI","MentHlth","PhysHlth"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for j,c in enumerate(cols):
    sns.histplot(df[c], bins=30, ax=axes[0,j], color="#4C78A8"); axes[0,j].set_title(f"Histogram {c}")
    sns.boxplot(x=df[c], ax=axes[1,j], color="#F58518"); axes[1,j].set_title(f"Boxplot {c}")
plt.tight_layout(); plt.savefig("../figures/02_hist_box.png", dpi=150); plt.show()
for c in cols:
    q1,q3 = df[c].quantile([.25,.75]); i = q3-q1
    n = ((df[c]<q1-1.5*i)|(df[c]>q3+1.5*i)).sum()
    print(f"{c}: min={df[c].min()}, median={df[c].median()}, max={df[c].max()}, ngoại lệ IQR={n} ({n/len(df):.2%})")
print("BMI>=60:", (df.BMI>=60).sum(), "| BMI>=80:", (df.BMI>=80).sum())

**Nhận xét:** BMI lệch phải (median 27, max 98), có 9.847 điểm ngoại lệ theo IQR (3,88%), trong đó 868 mẫu BMI ≥ 60 và 281 mẫu ≥ 80 là rất khó xảy ra thực tế. MentHlth và PhysHlth là số ngày (0–30), median bằng 0 và dồn ở 0 và 30; ngoại lệ IQR (14,3% và 16,1%) là người bệnh thật sự chứ không phải lỗi, nên không cắt.

## 2.4 Tỷ lệ mắc bệnh theo Age, Income, HighBP, GenHlth

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax,c in zip(axes, ["Age","Income","HighBP","GenHlth"]):
    r = df.groupby(c)[T].mean()*100
    sns.barplot(x=r.index.astype(int), y=r.values, ax=ax, color="#4C78A8")
    ax.set_title(f"Tỷ lệ mắc bệnh theo {c}"); ax.set_ylabel("% mắc bệnh")
plt.tight_layout(); plt.savefig("../figures/03_rate_by_group.png", dpi=150); plt.show()

**Nhận xét:** Tỷ lệ mắc tăng dần theo Age từ 1,4% (nhóm 1) lên khoảng 21,8% (nhóm 11) rồi giảm nhẹ ở nhóm 13 (18,5%). Income càng cao thì tỷ lệ càng thấp (24,3% → 8,0%). Người HighBP mắc 24,4% so với 6,0%, và GenHlth xấu (5) mắc tới 37,9% so với 2,5% ở mức tốt nhất.

## 2.5 Ma trận tương quan

In [ ]:
corr = df.corr()
plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size":6}, linewidths=.3)
plt.title("Ma trận tương quan (Pearson)"); plt.tight_layout()
plt.savefig("../figures/04_correlation.png", dpi=150); plt.show()
print(corr[T].drop(T).sort_values(key=abs, ascending=False).head(8).round(3))
f = corr.drop(index=T, columns=T)
pairs = f.where(np.triu(np.ones(f.shape), 1).astype(bool)).stack()
print(pairs.sort_values(key=abs, ascending=False).head(5).round(3))

**Nhận xét:** Tương quan với nhãn cao nhất là GenHlth (0,29), HighBP (0,26), DiffWalk và BMI (0,22); đều ở mức yếu–vừa nên không có biến nào đủ tự dự đoán. Cặp tương quan mạnh nhất giữa các thuộc tính chỉ là GenHlth–PhysHlth (0,52), PhysHlth–DiffWalk (0,48), Education–Income (0,45), đều dưới 0,7 nên không có đa cộng tuyến nghiêm trọng, chưa cần loại biến.